# X04 Transformer Feed-Forward Layers Are Key-Value Memories


## 目标

把 MLP 层解释成 key-value memory，是从 circuits 走向 facts、recall 和 model editing 的关键桥梁。


## 为什么现在学这篇

如果你只会读 attention 电路，不会解释 MLP 存了什么，就很难进一步理解 factual recall 和编辑。


## Notebook 与交付

- 原文: https://arxiv.org/abs/2012.14913
- Notebook：`notebooks/extensions/zh/x04_ffn_key_value_memories.ipynb`
- 先修: X01, M02
- 在 Colab 里复现一个教学版 key-value memory toy，并写 1 段说明为什么某些事实更像被 MLP 检索而不是被 attention 即时计算。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/learn-interpretability.git"
REPO_DIR = "learn-interpretability"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(candidate)],
            check=True,
        )
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import matplotlib.pyplot as plt
import numpy as np

countries = ["France", "Germany", "Japan"]
cities = ["Paris", "Berlin", "Tokyo"]
keys = np.eye(3)
values = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
])
query = np.array([1.0, 0.1, 0.0])
weights = np.exp(keys @ query)
weights = weights / weights.sum()
recalled = weights @ values

ablated_values = values.copy()
ablated_values[0] = 0.0
recalled_after_ablation = weights @ ablated_values

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(countries, weights, color="#1f5f8b")
axes[0].set_title("Key match weights")
axes[1].bar(cities, recalled, color="#c96a28", alpha=0.85, label="full")
axes[1].bar(cities, recalled_after_ablation, color="#5d8c63", alpha=0.65, label="ablate France value")
axes[1].set_title("Recalled value distribution")
axes[1].legend()
plt.tight_layout()

print("Top recalled city before ablation:", cities[int(np.argmax(recalled))])
print("Top recalled city after ablation:", cities[int(np.argmax(recalled_after_ablation))])


## 小结

MLP memory 视角让你能问“事实存在哪里”，而不是只问“信息从哪条路流过去”。


## Colab 优先的复现流程


### 运行前

- 在运行前先写 1 条你对机制的预测。
- 先写清你要对比的 baseline 是什么。
- 先决定什么结果会推翻你最喜欢的解释。


### 运行后

- 在笔记里把 observation 和 inference 分开。
- 标出复现之后仍然存在的 1 个歧义。
- 写 1 个能降低该歧义的下一步实验。


### 最后交付这些产物

- 在 Colab 里复现一个教学版 key-value memory toy，并写 1 段说明为什么某些事实更像被 MLP 检索而不是被 attention 即时计算。
- 1 份 experiment log，写清你改了哪些设置。
- 1 段“这次复现仍然不能证明什么”。


## 验收题

- 在你的 toy 里，什么对象扮演 key，什么对象扮演 value？
- 为什么把某个 recall 行为解释成 MLP 检索，会改变你下一步的实验设计？
- MLP memory 和 attention routing 在解释事实召回时各自更擅长说明什么？
- 如果你不能在不重开 notebook 的情况下独立答出至少 2 题，就回去重跑复现并重写 memo。
